In [109]:
import torch
import numpy as np
import pandas as pd
import joblib
import os

In [81]:
from torch import nn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from torch.utils.data import TensorDataset, DataLoader

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [19]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [69]:
X_data = pd.read_csv('/content/drive/MyDrive/tox (1)/X_final.csv')
y_data = pd.read_csv('/content/drive/MyDrive/tox (1)/y_final.csv')

In [70]:
print(f"X shape: {X_data.shape}")
print(f"y shape: {y_data.shape}")

X shape: (339055, 78)
y shape: (339055, 13)


In [72]:
X = X_data
y = y_data
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

X shape: (339055, 78)
y shape: (339055, 13)


In [75]:
RANDOM_STATE = 42
target_cols = y.columns.tolist()

mask = y[target_cols].notna().all(axis=1)
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

mask = y[target_cols].notna().all(axis=1)

X_clean = X_scaled[mask]
y_clean = y.loc[mask, target_cols].values

In [20]:
def masked_bce_loss(outputs, targets):
    mask = ~torch.isnan(targets)

    targets = torch.where(mask, targets, torch.zeros_like(targets))

    loss = nn.BCEWithLogitsLoss(reduction='none')(outputs, targets)

    loss = loss * mask
    return loss.sum() / mask.sum()

In [79]:
def train_and_validate(
    model,
    optimizer,
    train_loader,
    val_loader,
    num_epochs,
    device,
    verbose=True,
):
    train_losses, val_losses, val_rocs = [], [], []

    for epoch in range(1, num_epochs + 1):

        # ================= TRAIN =================
        model.train()
        running_loss = 0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            outputs = model(X_batch)
            loss = masked_bce_loss(outputs, y_batch)

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * X_batch.size(0)

        train_loss = running_loss / len(train_loader.dataset)
        train_losses.append(train_loss)

        # ================= VALID =================
        model.eval()
        running_loss = 0

        all_preds = []
        all_targets = []

        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)

                outputs = model(X_batch)
                loss = masked_bce_loss(outputs, y_batch)

                running_loss += loss.item() * X_batch.size(0)

                probs = torch.sigmoid(outputs)

                all_preds.append(probs.cpu().numpy())
                all_targets.append(y_batch.cpu().numpy())

        val_loss = running_loss / len(val_loader.dataset)
        val_losses.append(val_loss)

        # ================= ROC-AUC =================
        all_preds = np.vstack(all_preds)
        all_targets = np.vstack(all_targets)

        roc_scores = []

        for i in range(all_targets.shape[1]):
            mask = ~np.isnan(all_targets[:, i])

            if mask.sum() == 0:
                continue

            roc = roc_auc_score(
                all_targets[mask, i],
                all_preds[mask, i]
            )

            roc_scores.append(roc)

        mean_roc = np.mean(roc_scores)
        val_rocs.append(mean_roc)

        if verbose:
            print(
                f"Epoch {epoch} | "
                f"Train Loss={train_loss:.4f} | "
                f"Val Loss={val_loss:.4f} | "
                f"ROC-AUC={mean_roc:.4f}"
            )

    return train_losses, val_losses, val_rocs

In [22]:
class BaselineToxNet(nn.Module):
  def __init__(self, input_size, output_size):
        super(BaselineToxNet, self).__init__()

        self.fc1 = nn.Linear(input_size, 128)
        self.relu1 = nn.ReLU()

        self.dropout = nn.Dropout(0.2)

        self.fc2 = nn.Linear(128, 64)
        self.relu2 = nn.ReLU()

        self.fc3 = nn.Linear(64, output_size)

  def forward(self, x):
    out = self.fc1(x)
    out = self.relu1(out)
    out = self.dropout(out)
    out = self.fc2(out)
    out = self.relu2(out)
    out = self.dropout(out)
    out = self.fc3(out)
    return out

In [76]:
criterion= nn.BCEWithLogitsLoss()

In [83]:
X_train, X_val, y_train, y_val = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

X_train_torch = torch.tensor(X_train, dtype=torch.float32)
y_train_torch = torch.tensor(y_train[target_cols].values, dtype = torch.float32)

X_val_torch = torch.tensor(X_val, dtype=torch.float32)
y_val_torch = torch.tensor(y_val.values, dtype=torch.float32)


train_dataset = TensorDataset(X_train_torch, y_train_torch)
val_dataset = TensorDataset(X_val_torch, y_val_torch)


train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=512, shuffle=False)

In [85]:
model = BaselineToxNet(X_train.shape[1], y_train.shape[1]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)

epoch = 20
batch_size = 512


In [86]:
train_losses, val_losses, val_rocs = train_and_validate(
    model=model,
    optimizer=optimizer,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=20,
    device=device,
    verbose=True
)

Epoch 1 | Train Loss=0.2663 | Val Loss=0.2330 | ROC-AUC=0.7058
Epoch 2 | Train Loss=0.2368 | Val Loss=0.2246 | ROC-AUC=0.7432
Epoch 3 | Train Loss=0.2300 | Val Loss=0.2203 | ROC-AUC=0.7582
Epoch 4 | Train Loss=0.2262 | Val Loss=0.2171 | ROC-AUC=0.7652
Epoch 5 | Train Loss=0.2229 | Val Loss=0.2145 | ROC-AUC=0.7785
Epoch 6 | Train Loss=0.2201 | Val Loss=0.2121 | ROC-AUC=0.7820
Epoch 7 | Train Loss=0.2172 | Val Loss=0.2114 | ROC-AUC=0.7873
Epoch 8 | Train Loss=0.2150 | Val Loss=0.2092 | ROC-AUC=0.7879
Epoch 9 | Train Loss=0.2134 | Val Loss=0.2069 | ROC-AUC=0.7999
Epoch 10 | Train Loss=0.2116 | Val Loss=0.2064 | ROC-AUC=0.8016
Epoch 11 | Train Loss=0.2102 | Val Loss=0.2045 | ROC-AUC=0.8053
Epoch 12 | Train Loss=0.2088 | Val Loss=0.2052 | ROC-AUC=0.8053
Epoch 13 | Train Loss=0.2082 | Val Loss=0.2024 | ROC-AUC=0.8089
Epoch 14 | Train Loss=0.2063 | Val Loss=0.2026 | ROC-AUC=0.8080
Epoch 15 | Train Loss=0.2059 | Val Loss=0.2030 | ROC-AUC=0.8081
Epoch 16 | Train Loss=0.2048 | Val Loss=0.2025 | 

In [88]:
print("Final ROC-AUC:", val_rocs[-1])

Final ROC-AUC: 0.8091566909191427


In [89]:
class BatchmormToxNet(nn.Module):
  def __init__(self, input_size, output_size):
        super(BatchmormToxNet, self).__init__()

        self.fc1 = nn.Linear(input_size, 128)
        self.bn1 = nn.BatchNorm1d(128)
        self.relu1 = nn.ReLU()



        self.fc2 = nn.Linear(128, 64)
        self.bn2 = nn.BatchNorm1d(64)
        self.relu2 = nn.ReLU()

        self.dropout = nn.Dropout(0.2)

        self.fc3 = nn.Linear(64, output_size)

  def forward(self, x):
    out = self.fc1(x)
    out = self.bn1(out)
    out = self.relu1(out)
    out = self.dropout(out)

    out = self.fc2(out)
    out = self.bn2(out)
    out = self.relu2(out)
    out = self.dropout(out)

    out = self.fc3(out)
    return out

In [92]:
bn_model = BatchmormToxNet(X_train.shape[1], y_train.shape[1]).to(device)
optimizer = torch.optim.Adam(bn_model.parameters(), lr = 0.001)

epoch = 20
batch_size = 512


In [93]:
train_losses, val_losses, val_rocs = train_and_validate(
    model=bn_model,
    optimizer=optimizer,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=20,
    device=device,
    verbose=True
)

Epoch 1 | Train Loss=0.2728 | Val Loss=0.2254 | ROC-AUC=0.7561
Epoch 2 | Train Loss=0.2302 | Val Loss=0.2185 | ROC-AUC=0.7751
Epoch 3 | Train Loss=0.2243 | Val Loss=0.2147 | ROC-AUC=0.7914
Epoch 4 | Train Loss=0.2204 | Val Loss=0.2117 | ROC-AUC=0.7948
Epoch 5 | Train Loss=0.2178 | Val Loss=0.2099 | ROC-AUC=0.8043
Epoch 6 | Train Loss=0.2158 | Val Loss=0.2087 | ROC-AUC=0.8009
Epoch 7 | Train Loss=0.2141 | Val Loss=0.2071 | ROC-AUC=0.8109
Epoch 8 | Train Loss=0.2121 | Val Loss=0.2049 | ROC-AUC=0.8117
Epoch 9 | Train Loss=0.2107 | Val Loss=0.2044 | ROC-AUC=0.8140
Epoch 10 | Train Loss=0.2093 | Val Loss=0.2022 | ROC-AUC=0.8156
Epoch 11 | Train Loss=0.2083 | Val Loss=0.2023 | ROC-AUC=0.8134
Epoch 12 | Train Loss=0.2075 | Val Loss=0.2011 | ROC-AUC=0.8155
Epoch 13 | Train Loss=0.2063 | Val Loss=0.2012 | ROC-AUC=0.8154
Epoch 14 | Train Loss=0.2057 | Val Loss=0.2004 | ROC-AUC=0.8139
Epoch 15 | Train Loss=0.2044 | Val Loss=0.2001 | ROC-AUC=0.8158
Epoch 16 | Train Loss=0.2041 | Val Loss=0.1997 | 

In [94]:
def weighted_bce_with_logits(pos_weight):
    return nn.BCEWithLogitsLoss(pos_weight=pos_weight)

In [95]:
pos_weight = (y_train == 0).sum(axis=0) / (y_train == 1).sum(axis=0)
pos_weight = torch.tensor(pos_weight, dtype=torch.float32).to(device)

/tmp/ipykernel_4411/2559209719.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  pos_weight = torch.tensor(pos_weight, dtype=torch.float32).to(device)


In [96]:
class ResidualBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()

        self.block = nn.Sequential(
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
        )

        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(x + self.block(x))

In [105]:
class StrongToxNet(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()

        self.input = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.BatchNorm1d(256),
            nn.ReLU()
        )

        self.res1 = ResidualBlock(256)
        self.res2 = ResidualBlock(256)

        self.head = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, output_size)
        )

    def forward(self, x):
        out = self.input(x)
        out = self.res1(out)
        out = self.res2(out)
        out = self.head(out)
        return out

In [107]:
rb_model = StrongToxNet(X_train.shape[1], y_train.shape[1]).to(device)
optimizer = torch.optim.Adam(rb_model.parameters(), lr = 0.001)

epoch = 20
batch_size = 512

In [108]:
train_losses, val_losses, val_rocs = train_and_validate(
    model=rb_model,
    optimizer=optimizer,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=20,
    device=device,
    verbose=True
)

Epoch 1 | Train Loss=0.2420 | Val Loss=0.2190 | ROC-AUC=0.7622
Epoch 2 | Train Loss=0.2204 | Val Loss=0.2126 | ROC-AUC=0.7832
Epoch 3 | Train Loss=0.2117 | Val Loss=0.2048 | ROC-AUC=0.8047
Epoch 4 | Train Loss=0.2049 | Val Loss=0.2015 | ROC-AUC=0.8099
Epoch 5 | Train Loss=0.2003 | Val Loss=0.2000 | ROC-AUC=0.8090
Epoch 6 | Train Loss=0.1947 | Val Loss=0.1974 | ROC-AUC=0.8156
Epoch 7 | Train Loss=0.1912 | Val Loss=0.1979 | ROC-AUC=0.8164
Epoch 8 | Train Loss=0.1867 | Val Loss=0.1960 | ROC-AUC=0.8244
Epoch 9 | Train Loss=0.1826 | Val Loss=0.1968 | ROC-AUC=0.8181
Epoch 10 | Train Loss=0.1790 | Val Loss=0.1942 | ROC-AUC=0.8200
Epoch 11 | Train Loss=0.1754 | Val Loss=0.1925 | ROC-AUC=0.8249
Epoch 12 | Train Loss=0.1714 | Val Loss=0.1936 | ROC-AUC=0.8208
Epoch 13 | Train Loss=0.1685 | Val Loss=0.1945 | ROC-AUC=0.8209
Epoch 14 | Train Loss=0.1656 | Val Loss=0.1935 | ROC-AUC=0.8218
Epoch 15 | Train Loss=0.1617 | Val Loss=0.1942 | ROC-AUC=0.8240
Epoch 16 | Train Loss=0.1588 | Val Loss=0.1974 | 

In [113]:
models = {}
feature_names_dict = {}

for file in os.listdir('/content/drive/MyDrive/tox (1)/XGBoost_50/'):
    if file.endswith('.joblib'):
        cat = file.replace('xgb_', '').replace('.joblib', '')
        models[cat] = joblib.load(f'/content/drive/MyDrive/tox (1)/XGBoost_50/{file}')

        feature_names = models[cat].get_booster().feature_names
        feature_names_dict[cat] = feature_names


In [118]:
feature_idx = list(map(int, feature_names_dict['XGBoost_50_cardiotoxicity']))

In [119]:
X_val_top50 = X_val[:, feature_idx]

In [121]:
preds = {}

for cat, model in models.items():
    preds[cat] = model.predict_proba(X_val_top50)[:, 1]

In [130]:
xgb_preds = np.column_stack([
    preds[f"XGBoost_50_{cat}"] for cat in target_cols
])

In [139]:
X_val = X_val.values if hasattr(X_val, "values") else X_val
y_val = y_val[target_cols].values if hasattr(y_val, "values") else y_val

In [140]:
target_cols = list(target_cols)  # фиксируем порядок
n_tasks = len(target_cols)

In [141]:
target_cols = list(target_cols)  # фиксируем порядок
n_tasks = len(target_cols)

In [143]:
rb_model.eval()

all_preds = []

with torch.no_grad():
    for X_batch, y_batch in val_loader:
        X_batch = X_batch.to(device)

        outputs = rb_model(X_batch)        # logits
        probs = torch.sigmoid(outputs)   # вероятности

        all_preds.append(probs.cpu().numpy())

In [144]:
nn_preds = np.vstack(all_preds)

In [145]:
import numpy as np
from sklearn.metrics import roc_auc_score

weights = np.linspace(0, 1, 21)

best_w = {}
best_score = {}

for i, cat in enumerate(target_cols):

    y = y_val[:, i]
    mask = ~np.isnan(y)

    best_local_w = 0
    best_local_score = -1

    for w in weights:

        preds_ens = (
            w * xgb_preds[:, i] +
            (1 - w) * nn_preds[:, i]
        )

        score = roc_auc_score(y[mask], preds_ens[mask])

        if score > best_local_score:
            best_local_score = score
            best_local_w = w

    best_w[cat] = best_local_w
    best_score[cat] = best_local_score

    print(f"{cat}: w={best_local_w:.2f}, ROC-AUC={best_local_score:.4f}")

acute_toxicity: w=1.00, ROC-AUC=0.9451
carcinogenicity: w=1.00, ROC-AUC=0.9808
cardiotoxicity: w=1.00, ROC-AUC=0.9858
dermal_toxicity: w=0.95, ROC-AUC=0.9824
genotoxicity: w=1.00, ROC-AUC=0.9917
hepatotoxicity: w=0.85, ROC-AUC=0.9797
ocular_toxicity: w=1.00, ROC-AUC=0.9769
oxidative_stress: w=1.00, ROC-AUC=0.9420
respiratory_toxicity: w=1.00, ROC-AUC=0.9731
neuro_sensory_toxicity: w=0.85, ROC-AUC=0.9897
immuno_hematotoxicity: w=1.00, ROC-AUC=0.9471
reprod_dev_toxicity: w=0.35, ROC-AUC=0.9663
endocrine_metabolic_tox: w=1.00, ROC-AUC=0.9244


In [147]:
torch.save(rb_model.state_dict(), "rb_model.pt")

In [152]:
load_model = StrongToxNet(X_train.shape[1], y_train.shape[1]).to(device)

In [153]:
load_model.load_state_dict(torch.load("rb_model.pt", map_location=device))
load_model.eval()

StrongToxNet(
  (input): Sequential(
    (0): Linear(in_features=78, out_features=256, bias=True)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
  )
  (res1): ResidualBlock(
    (block): Sequential(
      (0): Linear(in_features=256, out_features=256, bias=True)
      (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): Dropout(p=0.2, inplace=False)
      (4): Linear(in_features=256, out_features=256, bias=True)
      (5): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (relu): ReLU()
  )
  (res2): ResidualBlock(
    (block): Sequential(
      (0): Linear(in_features=256, out_features=256, bias=True)
      (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): Dropout(p=0.2, inplace=False)
      (4): Linear(in_features=256, out_features=256, bias=True)
      (5): Ba